# Training NPClassifier Models

This notebook walks through **two training workflows** using `torch_np_classifier`:

1. **Single flat model** — one `NPClassifierLightning` that predicts all 730 classes (pathway + superclass + class) in one shot.
2. **Three hierarchical models** — separate models for each level (7 pathway / 70 superclass / 653 class), which are later combined in an `NPClassifierEnsemble` for voting-based prediction.

**Prerequisites**
- `train_dataset.csv`, `validation_dataset.csv`, `test_dataset.csv` in `../examples/`
- Each CSV must have columns: `key`, `SMILES`, then one binary column per class label.

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
import torch
import lightning as L
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from torch_np_classifier import (
    NPClassifierFeaturizer,
    NPClassifierDataModule,
    NPClassifierLightning,
)

## 1. Load datasets

In [ ]:
TRAIN_CSV = "../examples/train_dataset.csv"
VAL_CSV = "../examples/validation_dataset.csv"
TEST_CSV = "../examples/test_dataset.csv"

df_train = pd.read_csv(TRAIN_CSV)
df_val = pd.read_csv(VAL_CSV)
df_test = pd.read_csv(TEST_CSV)

# Feature columns are everything after key + SMILES
label_cols = [c for c in df_train.columns if c not in ("key", "SMILES")]
label_names = np.array(label_cols)

print(f"Train rows : {len(df_train):,}")
print(f"Val rows   : {len(df_val):,}")
print(f"Test rows  : {len(df_test):,}")
print(f"Label count: {len(label_names)}  (pathway 7 | superclass 70 | class 653)")
df_train[label_cols].sum().sort_values(ascending=False).head(10)

## 2. Featurize SMILES → Morgan fingerprints (6 144-dim)

`NPClassifierFeaturizer` computes the NP-Classifier fingerprint:
- 3 × 2 048-bit Morgan fingerprints at radii 0, 1, 2 (binary)
- packed into a single 6 144-dimensional float32 vector.

In [ ]:
featurizer = NPClassifierFeaturizer(radius=2, n_jobs=-1)

X_train = featurizer.transform(df_train["SMILES"].tolist())
X_val = featurizer.transform(df_val["SMILES"].tolist())
X_test = featurizer.transform(df_test["SMILES"].tolist())

y_train = df_train[label_cols].values.astype(np.float32)
y_val = df_val[label_cols].values.astype(np.float32)
y_test = df_test[label_cols].values.astype(np.float32)

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")

## 3. Single flat model (all 730 labels)

A single model predicts all 730 NP-Classifier classes simultaneously.  
Good for quick experiments — the ensemble approach in section 4 gives finer control.

In [ ]:
N_LABELS = y_train.shape[1]  # 730

dm = NPClassifierDataModule(
    X_train,
    y_train,
    X_val=X_val,
    y_val=y_val,
    batch_size=128,
    num_workers=4,
)

flat_model = NPClassifierLightning(
    input_dim=6144,
    num_classes=N_LABELS,
    hidden_dims=[2048, 1024, 512],
    dropout=0.3,
    learning_rate=1e-3,
)

callbacks = [
    EarlyStopping("val_loss", patience=5, mode="min"),
    ModelCheckpoint(
        dirpath="checkpoints/flat",
        filename="np_classifier_flat",
        monitor="val_loss",
        mode="min",
        save_top_k=1,
    ),
]

trainer = L.Trainer(
    max_epochs=50,
    callbacks=callbacks,
    log_every_n_steps=10,
    accelerator="auto",
)
trainer.fit(flat_model, dm)

### 3.1 Evaluate on test set

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


def predict_proba(model, X):
    model.eval()
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = DataLoader(ds, batch_size=256, shuffle=False)
    t = L.Trainer(enable_progress_bar=False, logger=False, accelerator="auto")
    raw = t.predict(model, loader)
    return torch.cat(raw, dim=0).numpy()


probs_test = predict_proba(flat_model, X_test)
preds_test = (probs_test >= 0.5).astype(int)

# Level slices
PATHWAY_SLICE = slice(0, 7)
SUPERCLASS_SLICE = slice(7, 77)
CLASS_SLICE = slice(77, None)

for name, sl in [
    ("pathway", PATHWAY_SLICE),
    ("superclass", SUPERCLASS_SLICE),
    ("class", CLASS_SLICE),
]:
    f1 = f1_score(y_test[:, sl], preds_test[:, sl], average="macro", zero_division=0)
    print(f"  {name:12s}  macro-F1 = {f1:.4f}")

## 4. Three hierarchical models

Training one model per level of the hierarchy gives the ensemble's voting algorithm more signal.  
Each model is smaller and trains faster than the flat 730-class model.

In [ ]:
import os

LEVELS = {
    "pathway": (slice(0, 7), 7),
    "superclass": (slice(7, 77), 70),
    "class": (slice(77, None), 653),
}

trained_models = {}

for level, (sl, n_out) in LEVELS.items():
    print(f"\n{'=' * 50}")
    print(f"Training  {level}  ({n_out} outputs)")
    print("=" * 50)

    dm_lvl = NPClassifierDataModule(
        X_train,
        y_train[:, sl],
        X_val=X_val,
        y_val=y_val[:, sl],
        batch_size=128,
        num_workers=4,
    )

    model_lvl = NPClassifierLightning(
        input_dim=6144,
        num_classes=n_out,
        hidden_dims=[2048, 1024, 512],
        dropout=0.3,
        learning_rate=1e-3,
    )

    ckpt_dir = f"checkpoints/hierarchical/{level}"
    os.makedirs(ckpt_dir, exist_ok=True)

    cbs = [
        EarlyStopping("val_loss", patience=5, mode="min"),
        ModelCheckpoint(
            dirpath=ckpt_dir,
            filename=f"{level}_np_classifier",
            monitor="val_loss",
            mode="min",
            save_top_k=1,
        ),
    ]

    t = L.Trainer(
        max_epochs=50,
        callbacks=cbs,
        log_every_n_steps=10,
        accelerator="auto",
    )
    t.fit(model_lvl, dm_lvl)
    trained_models[level] = model_lvl

### 4.1 Evaluate each level on the test set

In [ ]:
for level, (sl, _) in LEVELS.items():
    probs = predict_proba(trained_models[level], X_test)
    preds = (probs >= 0.5).astype(int)
    f1 = f1_score(y_test[:, sl], preds, average="macro", zero_division=0)
    print(f"  {level:12s}  macro-F1 = {f1:.4f}")

### 4.2 Reload from checkpoints (optional)

If you come back to this notebook after training, load models directly from disk.

In [ ]:
# Reload from the best checkpoints saved above
pathway_model = NPClassifierLightning.load_from_checkpoint(
    "checkpoints/hierarchical/pathway/pathway_np_classifier.ckpt"
)
superclass_model = NPClassifierLightning.load_from_checkpoint(
    "checkpoints/hierarchical/superclass/superclass_np_classifier.ckpt"
)
class_model = NPClassifierLightning.load_from_checkpoint(
    "checkpoints/hierarchical/class/class_np_classifier.ckpt"
)

print("Models loaded from checkpoints.")